In [ ]:
 # 1. Install semua library yang dibutuhkan
!pip install -q transformers peft librosa torch evaluate jiwer pandas matplotlib seaborn datasets

import pandas as pd
import numpy as np
import librosa
import torch
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datasets import Dataset

In [ ]:
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

In [ ]:
import tarfile

# 2. Tentukan Path
PATH_ZIP_DI_DRIVE = "/content/drive/MyDrive/PROYEK DATA MINING/1774204205957-cv-corpus-25.0-2026-03-09-id.tar.gz"
EXTRACT_PATH = "/content/dataset_temp"

# 3. Ekstrak Data (Lewati jika sudah diekstrak sebelumnya)
if not os.path.exists(EXTRACT_PATH):
    print("Mengekstrak dataset...")
    with tarfile.open(PATH_ZIP_DI_DRIVE, "r:gz") as tar:
        tar.extractall(path=EXTRACT_PATH)
    print("Ekstrak Selesai!")

# 4. Tentukan path TSV dan folder audio
TSV_PATH = os.path.join(EXTRACT_PATH, "cv-corpus-25.0-2026-03-09", "id", "validated.tsv")
AUDIO_FOLDER = os.path.join(EXTRACT_PATH, "cv-corpus-25.0-2026-03-09", "id", "clips")

# 5. Baca Metadata
df = pd.read_csv(TSV_PATH, sep='\t')
df_with_accent = df.dropna(subset=['accents'])
print(f"Total data dengan label aksen: {len(df_with_accent)}")

In [ ]:
# 1. FUNGSI AUGMENTASI AUDIO (Adaptive Synthetic)
def apply_audio_augmentation(audio_array, sr=16000):
    """Memberikan efek augmentasi acak pada audio untuk mencegah overfitting"""
    aug_type = np.random.choice(['pitch', 'speed', 'noise', 'none'])

    if aug_type == 'pitch':
        return librosa.effects.pitch_shift(audio_array, sr=sr, n_steps=np.random.uniform(-2, 2))
    elif aug_type == 'speed':
        return librosa.effects.time_stretch(audio_array, rate=np.random.uniform(0.85, 1.15))
    elif aug_type == 'noise':
        noise = np.random.randn(len(audio_array))
        return audio_array + 0.005 * noise
    else:
        return audio_array

# 2. PROSES BALANCING DATASET (Target: Semua aksen rata 150 data train, 15 data test)
target_jumlah_data = 150
top_accents = df_with_accent['accents'].value_counts().nlargest(4).index.tolist()

train_list = []
test_list = []

for accent in top_accents:
    df_aksen = df_with_accent[df_with_accent['accents'] == accent]

    # Data Uji (Test) Murni, tanpa augmentasi
    df_test = df_aksen.sample(15, random_state=42)
    test_list.append(df_test)

    # Sisa untuk Pelatihan (Train)
    df_train_sisa = df_aksen.drop(df_test.index)

    # Jika data kurang (Minoritas), lakukan oversampling + tandai augmentasi
    if len(df_train_sisa) < target_jumlah_data:
        kekurangan = target_jumlah_data - len(df_train_sisa)
        df_tambahan = df_train_sisa.sample(kekurangan, replace=True, random_state=42).copy()
        df_tambahan['is_augmented'] = True
        df_train_sisa['is_augmented'] = False
        train_list.append(pd.concat([df_train_sisa, df_tambahan]))
    # Jika data berlebih (Mayoritas/Betawi), lakukan undersampling
    else:
        df_train_sisa = df_train_sisa.sample(target_jumlah_data, random_state=42).copy()
        df_train_sisa['is_augmented'] = False
        train_list.append(df_train_sisa)

df_train_final = pd.concat(train_list).reset_index(drop=True)
df_test_final = pd.concat(test_list).reset_index(drop=True)

print("Proses Balancing Selesai!")

In [ ]:
# Tampilkan rincian
rincian_aug = df_train_final.groupby(['accents', 'is_augmented']).size().unstack(fill_value=0)
rincian_aug.columns = ['Asli (Original)', 'Sintetik (Augmented)'] if len(rincian_aug.columns) == 2 else ['Asli (Original)']
print("=== RINCIAN DATA ASLI VS AUGMENTASI SINTETIK ===")
print(rincian_aug)

# Buat Grafik
plt.figure(figsize=(10, 6))
sns.countplot(data=df_train_final, x='accents', hue='is_augmented', palette='viridis')
plt.title('Distribusi Dataset Setelah Penyeimbangan (Balancing & Augmentasi)')
plt.xlabel('Aksen / Dialek')
plt.ylabel('Jumlah Audio')
plt.legend(title='Status Data', labels=['Asli', 'Augmentasi (Sintetik)'])
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
from transformers import WhisperProcessor
import librosa
import os
from datasets import Dataset

model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id, language="indonesian", task="transcribe")

def prepare_dataset(batch):
    path = os.path.join(AUDIO_FOLDER, batch["path"])
    audio, _ = librosa.load(path, sr=16000)

    # Eksekusi Augmentasi On-The-Fly jika ditandai
    if batch.get('is_augmented', False):
        audio = apply_audio_augmentation(audio)

    batch["input_features"] = processor.feature_extractor(audio, sampling_rate=16000).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

print("Mengekstrak fitur audio Train...")
train_dataset = Dataset.from_pandas(df_train_final).map(
    prepare_dataset,
    remove_columns=df_train_final.columns.tolist()
)

print("Mengekstrak fitur audio Test...")
test_dataset = Dataset.from_pandas(df_test_final).map(
    prepare_dataset,
    remove_columns=df_test_final.columns.tolist()
)

print("Ekstraksi fitur selesai!")

In [ ]:
import torch
!pip install --upgrade torchao
from transformers import AutoModelForSpeechSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model
from dataclasses import dataclass
from typing import Any, Dict, List, Union

# 1. Load Model
model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, device_map="auto")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# 2. SpecAugment (Regulasi Mencegah Overfitting)
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.1
model.config.mask_feature_prob = 0.1

# 3. Setup LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,  # Dinaikkan untuk regulasi
    bias="none"
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()

# 4. Data Collator
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
# Install MLflow
!pip install -q mlflow

import mlflow
import os

# Mengatur environment MLflow untuk logging lokal
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///mlflow.db"
mlflow.set_experiment("Whisper_LoRA_Accent_Optimization")

# MLflow akan otomatis mendeteksi Trainer dari HuggingFace
# dan melakukan logging parameter serta metrik.
%load_ext google.colab.data_table

In [ ]:
# Training Arguments dengan Weight Decay (for Overfitting)
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-lora-balanced",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    weight_decay=0.01,           # <--- Kunci cegah overfitting
    warmup_steps=50,
    max_steps=300,
    fp16=True,
    eval_strategy="steps",
    eval_steps=30,
    save_steps=30,
    load_best_model_at_end=True, # <--- Otomatis pilih bobot terbaik
    metric_for_best_model="loss",
    greater_is_better=False,
    logging_steps=10
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Berhenti jika loss tidak membaik
)

print("\n=== MEMULAI TRAINING MODEL ===")
trainer.train()

# Simpan model final
output_final_dir = "./whisper-lora-final"
trainer.save_model(output_final_dir)
processor.save_pretrained(output_final_dir)
print(f"\nModel berhasil disimpan di: {output_final_dir}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Mengambil riwayat log otomatis dari memori Trainer HuggingFace
log_history = trainer.state.log_history

# 2. Memisahkan data metrik Training Loss dan Validation (Eval) Loss
train_logs = [log for log in log_history if 'loss' in log and 'step' in log]
eval_logs = [log for log in log_history if 'eval_loss' in log and 'step' in log]

# 3. Mengubah menjadi DataFrame agar mudah divisualisasikan
df_train = pd.DataFrame(train_logs)
df_eval = pd.DataFrame(eval_logs)

# 4. Membuat Visualisasi Grafik
plt.figure(figsize=(10, 6))

# Plot Training Loss (jika ada)
if not df_train.empty:
    plt.plot(df_train['step'], df_train['loss'], marker='o', linewidth=2,
             label='Training Loss', color='#1f77b4', linestyle='-')

# Plot Validation Loss (jika ada)
if not df_eval.empty:
    plt.plot(df_eval['step'], df_eval['eval_loss'], marker='s', linewidth=2,
             label='Validation Loss', color='#d62728', linestyle='--')

# 5. Kostumisasi Tampilan
plt.title('Kurva Pembelajaran (Learning Curve) Real-Time', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Training Steps', fontsize=12, fontweight='bold')
plt.ylabel('Loss Value', fontsize=12, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.7)

if not df_train.empty:
    plt.xticks(df_train['step'])

plt.tight_layout()
plt.show()

In [ ]:
from tqdm.auto import tqdm

wer_metric = evaluate.load("wer")
model.eval()
hasil_evaluasi = []

# Reset config yang menyebabkan error agar menggunakan generation_config
model.config.suppress_tokens = None

print("\n=== MEMULAI EVALUASI PER AKSEN ===")
for accent in top_accents:
    df_uji_aksen = df_test_final[df_test_final['accents'] == accent]
    prediksi_teks = []
    referensi_teks = []

    print(f"Mengevaluasi aksen: {accent}...")
    for _, row in tqdm(df_uji_aksen.iterrows(), total=len(df_uji_aksen)):
        audio, _ = librosa.load(os.path.join(AUDIO_FOLDER, row['path']), sr=16000)
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features.to("cuda")

        with torch.no_grad():
            # Gunakan parameter langsung di generate untuk menghindari konflik config
            generated_ids = model.generate(inputs, language="indonesian", task="transcribe")

        hasil_transkrip = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        prediksi_teks.append(hasil_transkrip.lower())
        referensi_teks.append(row['sentence'].lower())

    nilai_wer = wer_metric.compute(predictions=prediksi_teks, references=referensi_teks)
    hasil_evaluasi.append({"Aksen / Dialek": accent, "Word Error Rate (WER)": round(nilai_wer, 4)})

# Tampilkan Tabel Evaluasi
df_laporan_evaluasi = pd.DataFrame(hasil_evaluasi)
print("\n=== HASIL EVALUASI WHISPER-LORA SETELAH OPTIMASI ===")
print(df_laporan_evaluasi.to_string(index=False))

# Visualisasi Bar Chart WER
plt.figure(figsize=(9, 5))
sns.barplot(x='Aksen / Dialek', y='Word Error Rate (WER)', data=df_laporan_evaluasi, palette='Reds_r')
plt.title('Evaluasi Akhir: Word Error Rate (WER) per Aksen')
plt.ylabel('Nilai WER (Semakin Rendah Semakin Baik)')
plt.show()

In [ ]:
import gradio as gr
import torch
import librosa

# --- AGEN LOGIKA: Fungsi Deteksi Aksen Berbasis Teks ---
def deteksi_aksen(teks):
    teks_lower = teks.lower()

    # Kamus penanda aksen/dialek
    jawa_markers = ["kulo", "jenengan", "nggih", "mboten", "sampun", "monggo", "nderek", "lho", "toh"]
    betawi_markers = ["gue", "lu", "elu", "banget", "kagak", "nggak", "nyak", "babe", "entar", "dulu"]
    sunda_markers = ["atuh", "punten", "kumaha", "damang", "pisan", "mah", "teh", "saha", "euy"]

    # Menghitung kecocokan
    skor_jawa = sum(1 for kata in jawa_markers if kata in teks_lower)
    skor_betawi = sum(1 for kata in betawi_markers if kata in teks_lower)
    skor_sunda = sum(1 for kata in sunda_markers if kata in teks_lower)

    if skor_jawa > skor_betawi and skor_jawa > skor_sunda:
        return "Jawa / Medhok (Terdeteksi dari pola kosakata)"
    elif skor_betawi > skor_jawa and skor_betawi > skor_sunda:
        return "Betawi / Jakarta (Terdeteksi dari pola kosakata)"
    elif skor_sunda > skor_jawa and skor_sunda > skor_betawi:
        return "Sunda (Terdeteksi dari pola kosakata)"
    else:
        return "Standar / Tidak Terdeteksi Spesifik (Netral)"

# --- FUNGSI INFERENSI UTAMA ---
def analyze_transcription(audio_path, pilihan_bahasa):
    if audio_path is None:
        return "Silakan rekam audio terlebih dahulu.", ""

    try:
        audio, _ = librosa.load(audio_path, sr=16000)
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features.to("cuda")

        with torch.no_grad():
            generated_ids = model.generate(inputs, language=pilihan_bahasa, task="transcribe")

        teks_asli = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # Eksekusi Deteksi Aksen
        if teks_asli.strip():
            estimasi_aksen = deteksi_aksen(teks_asli)
        else:
            estimasi_aksen = "Suara tidak terdeteksi."

        return teks_asli, estimasi_aksen

    except Exception as e:
        return f"Error: {str(e)}", "Gagal memproses."

# --- ANTARMUKA (UI) GRADIO ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ Dashboard Analisis Pola Kesalahan & Deteksi Aksen (Whisper-LoRA)")
    gr.Markdown("Pilih parameter bahasa dan unggah/rekam suara untuk menganalisis akurasi transkripsi model.")

    with gr.Row():
        with gr.Column():
            input_bahasa = gr.Dropdown(choices=["indonesian", "javanese", "sundanese"], value="indonesian", label="Paksa Aturan Bahasa (Language Parameter)")
            audio_input = gr.Audio(type="filepath", label="Input Suara")
            submit_btn = gr.Button("Analisis Suara", variant="primary")

        with gr.Column():
            output_transkripsi = gr.Textbox(label="1️⃣ Hasil Transkripsi Model", lines=4)
            output_aksen = gr.Textbox(label="2️⃣ Deteksi Aksen", lines=2)

    submit_btn.click(
        fn=analyze_transcription,
        inputs=[audio_input, input_bahasa],
        outputs=[output_transkripsi, output_aksen]
    )

demo.launch(share=True)